## Demographic Validation of EV Cohort and entire user population
Author: Callie, Anne, Salsabil

In [ ]:
!pip install seaborn

import pandas as pd
import matplotlib.pyplot as plt
import os
import geopandas as gpd
import seaborn as sns
import numpy as np
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import scipy.stats as stats
from scipy.stats import linregress


  Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)


In [ ]:
#FOR LOADING FROM SPECTUS
root = '/home/jovyan/SAI_EVCS/'
exec(open(root+'scripts/EV00_settings.py').read())


start_date = "20220301"
end_date = "20220531"
current_city = "Denver"

with open(current_session_path+f'ev_driver_info/user_set_{current_city}_model.txt', "r") as file:
    set_string = file.read() # Read the string from the file
ev_driver_ls = list(ast.literal_eval(set_string)) # Convert the string back to a set


In [ ]:
#FOR LOADING FROM SPECTUS
root = '/home/jovyan/SAI_EVCS/'


#user info
user_csv =current_session_path+ 'all_driver_info/'
user_1 = pd.read_csv(user_csv+f'all_driver_info_{current_city}_2022_3_cbg_demographics.csv',index_col=0)
user_2 = pd.read_csv(user_csv+f'all_driver_info_{current_city}_2022_4_cbg_demographics.csv',index_col=0)
user_3 = pd.read_csv(user_csv+f'all_driver_info_{current_city}_2022_5_cbg_demographics.csv',index_col=0)
user_= pd.concat([user_1,user_2,user_3])
user_info=user_.drop_duplicates(subset='cuebiq_id')
#user_info.dropna(inplace=True)




user_info.rename(columns={
    'perc_white': 'percent_white',
    'perc_black': 'percent_black',
    'perc_hisp': 'percent_hispanic',
    'perc_poverty': 'percent_poverty',
    'income_median': 'medincome',
    'perc_sfh_owners': 'percent_sfh_owners',
    'perc_mfh_renters': 'percent_mfh_renters',
    'vmt' : 'VMT',
    'perc_ev' : 'CEC_BEV_per', #TO FILL NAME FOR THE % of EV in CBG
    'home_GEOID' : 'GEOID' #TO FILL NAME FOR THE GEOID/CBG
}, inplace=True)



In [ ]:
print(f" unique cuebiq_id in user_info (all bay area):",user_info['cuebiq_id'].nunique())
print(f" unique cuebiq_id in ev driver list:",len(set(ev_driver_ls)))
print(f" EV Ownership Rate",round(len(set(ev_driver_ls))/user_info['cuebiq_id'].nunique(),4))

In [ ]:
cols=['cuebiq_id','percent_white', 'percent_black', 'percent_hispanic','percent_poverty', 'perc_asian','VMT', 'medincome', 'percent_sfh_owners','percent_mfh_renters', 'CEC_BEV_per']


from scipy.stats import mannwhitneyu
cbg_user = user_info.drop_duplicates(subset='cuebiq_id')[cols]
cbg_user=cbg_user.dropna(how='all')

cbg_ev=cbg_user[cbg_user.cuebiq_id.isin(ev_driver_ls)]


# Initialize results storage
stat_results = []

# Statistical comparison and visualization
for var in ['VMT', 'medincome','percent_poverty','percent_sfh_owners',
    'percent_mfh_renters', 'percent_white', 'perc_asian','percent_black', 'percent_hispanic',
     ]:
    stat, p = mannwhitneyu(cbg_user[f"{var}"].dropna(), cbg_ev[f"{var}"].dropna(), alternative='two-sided')
    stat_results.append({
        "Variable": var,
        "All CBGs": cbg_user[f"{var}"].mean(),
        "EV CBGs": cbg_ev[f"{var}"].mean(),
        "p-value": p
    })

# Convert results to DataFrame for display
results_df = pd.DataFrame(stat_results)
display(results_df.round(3))
# Show side-by-side KDE plots for a few key variables
key_vars = ['percent_white', 'percent_black', 'percent_hispanic','perc_asian','CEC_BEV_per',
            'percent_poverty', 'medincome', 'percent_sfh_owners', 'percent_mfh_renters','VMT']
plt.figure(figsize=(15, 8))

for i, var in enumerate(key_vars):
    plt.subplot(3, 4, i+1)
    sns.kdeplot(cbg_user[var], label='All Users', fill=True)
    sns.kdeplot(cbg_ev[var], label='EV Drivers', fill=True)
    plt.title(var)
    plt.legend()

plt.tight_layout()
plt.show()